This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
context = gx.get_context()
import logging

In [2]:
logging.basicConfig(level=logging.INFO, force = True)

In [3]:
## THIS IS REQUIRED FOR THE TECHNICAL VIEW HACK
# TODO handling of credentials not ideal, required for technical view fix
import os

from google.cloud import bigquery
import pandas as pd

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json'
client = bigquery.Client()

In [4]:
gx_temp_schema = "tech_great_expectations_temp_ttl_7d"
date_filter = "hour <= '2099-12-31'"

In [5]:
def partition_enforcement_enabled(client, schema_name: str, table_name: str):
    partition_enforcement_enabled_sql = f"""
  SELECT
    option_value
  FROM
    {schema_name}.INFORMATION_SCHEMA.TABLE_OPTIONS
  WHERE
    table_name = '{table_name}'
  AND 
    option_name = 'require_partition_filter';
    """
    
    # TODO CHO20230622 handle exceptions
    partition_enforcement_enabled_res = pd.read_gbq(partition_enforcement_enabled_sql, project_id='world-fishing-827', dialect='standard')

    # TODO CHO20230622 handle multiple rows returned
    return(partition_enforcement_enabled_res.shape[0] > 0)

In [6]:
def get_partition_cols(client, schema_name: str, table_name: str):
    get_partition_cols_sql = f"""
  SELECT
    column_name,
    data_type
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    is_partitioning_column = 'YES';
    """
    
    # TODO CHO20230622 handle exceptions
    get_partition_cols_res = pd.read_gbq(get_partition_cols_sql, project_id='world-fishing-827', dialect='standard')

    # TODO CHO20230622 handle multiple rows returned
    return(get_partition_cols_res)

In [7]:
def create_or_replace_tech_view(
    client, 
    dataset_name: str, 
    table_name: str, 
    gx_temp_schema: str, 
    default_max_date: str = "2099-12-31", 
    date_filter: str = None,
    partition_cols: str = None,
    partition_filter_enforced = None
):
    # TODO CHO20230623 automatically get partition col
    if date_filter is None:
        if partition_filter_enforced is None:
            partition_filter_enforced = partition_enforcement_enabled(client, dataset_name, table_name)
        if partition_filter_enforced:
            if partition_cols is None:
                partition_cols = get_partition_cols(client, dataset_name, table_name)
            if partition_cols.query("data_type.isin(['TIMESTAMP', 'DATE'])").shape[0]:
                partition_col_used = partition_cols.query("data_type.isin(['TIMESTAMP', 'DATE'])")["column_name"][0]
                date_filter = f"{partition_col_used} < '{default_max_date}'"
                logging.info(f"""
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: {date_filter}
                """)
            else:
                logging.warn(f"""
                Partition enforcement enabled but no date columns found among partitions columns {partition_date_cols['column_name']}.
                Proceeding but eventually queries will fail!
                """)
        else:
            logging.info(f"Partition enforcement not enabled, not applying date filter")
    else:
        logging.info(f"Using provided date filter: {date_filter}")

    date_filter_sql = "" if date_filter is None else f" WHERE {date_filter}"
    # TODO CHO20230622 validate parameters, e.g. date filter
    # TODO CHO20230622 check first whether view exists before replacing
    fq_view_name = f"{gx_temp_schema}.v_unfiltered_{dataset_name}_{table_name}"
    query = f"""
    CREATE OR REPLACE VIEW `{fq_view_name}` AS (
    SELECT * FROM {dataset_name}.{table_name} {date_filter_sql}
    )
    """

    # TODO CHO20230622 handle exceptions
    client.query(query)
    return(fq_view_name)    

In [8]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [9]:
import yaml

In [10]:
with open("great_expectations/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [11]:
datasource_config.get("project")

'gfw-google-827'

In [12]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

In [13]:
def add_datasource(client, dataset_name, table_name, gx_temp_schema, gx_datasource):
    partition_cols = get_partition_cols(client, dataset_name, table_name)
    partition_filter_enforced = partition_enforcement_enabled(client, dataset_name, table_name)
    gx_view_name = create_or_replace_tech_view(
        client=client, 
        dataset_name=dataset_name, 
        table_name=table_name, 
        gx_temp_schema=gx_temp_schema, 
        partition_cols=partition_cols, 
        partition_filter_enforced=partition_filter_enforced
    )
    if datasource_name in gx_datasource.get_asset_names():
        table_asset = gx_datasource.get_asset(datasource_name)
    else:
        table_asset = gx_datasource.add_query_asset(
            name=datasource_name,
            query=f"SELECT * FROM {gx_view_name}"
        )
        
    # add datetime splitter and sorter if datetime partition column exists
    if partition_cols.query("data_type.isin(['TIMESTAMP', 'DATE'])").shape[0]:
        date_time_partition_col_used = partition_cols\
            .query("data_type.isin(['TIMESTAMP', 'DATE'])")["column_name"][0]
        
        table_asset.add_splitter_column_value(date_time_partition_col_used)
        table_asset.add_sorters([f"-{date_time_partition_col_used}"])

In [14]:
for current_datasource in datasource_config.get("datasources"):
    print(f"Loading datasource '{current_datasource.get('name')}'")
    datasource_name = current_datasource.get('name')
    latest_version = current_datasource.get('latest_version')
    config_latest_version = current_datasource.get('versions').get(latest_version)
    dataset_name = config_latest_version.get('dataset')
    table_name = config_latest_version.get('table')
    add_datasource(client, dataset_name, table_name, gx_temp_schema, gx_datasource)    

Loading datasource 'messages'


INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: timestamp < '2099-12-31'
                
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Loading datasource 'satellite_timing_offsets'


INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: hour < '2099-12-31'
                
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Loading datasource 'segs_activity'


...

INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: date < '2099-12-31'
                
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Loading datasource 'stats_daily'


INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: date < '2099-12-31'
                
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Loading datasource 'vessel_info'


INFO:root:Partition enforcement not enabled, not applying date filter
INFO:great_expectations.data_context.data_context.file_data_context:Saving 2 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


In [15]:
gx_datasource.get_asset_names()

{'messages',
 'satellite_timing_offsets',
 'segs_activity',
 'segs_activity_daily',
 'ssvids_identities',
 'ssvids_identities_daily',
 'stats_daily',
 'vessel_info'}